In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

from config import DOSSIER_CP, CATALOGUE, LLM_MODEL , GEN_MODEL
print(DOSSIER_CP, DOSSIER_CP.exists())

C:\Users\pc\Desktop\Mes Peojets\rag-assurance\data\cp True


In [2]:
from pypdf import PdfReader

if not DOSSIER_CP.exists() or not any(DOSSIER_CP.iterdir()):
    print("Le dossier est vide ou inexistant :", DOSSIER_CP)
    
for fichier in sorted(DOSSIER_CP.glob("*.pdf")):
    print(fichier.name)



cp_01_luko_etendue.pdf
cp_02_luko_minimum.pdf
cp_03_sogessur_F190315.pdf
cp_04_gmf_domopass.pdf
cp_05_matmut_residence_principale.pdf
cp_06_axa_hors_base.pdf


In [12]:
texte_cp3 = PdfReader(DOSSIER_CP / "cp_03_sogessur_F190315.pdf").pages[0].extract_text()
texte_cp4 = PdfReader(DOSSIER_CP / "cp_04_gmf_domopass.pdf").pages[0].extract_text()

# print("\n" + texte_cp3)
# print("\n" + texte_cp4)

In [13]:
# from typing import Optional
# from pydantic import BaseModel, Field

# class FicheCP(BaseModel):
#     assureur: str = Field(description="Nom de l'assureur tel qu'écrit dans le document")
#     reference_cg: Optional[str] = Field(description="Référence des Conditions Générales applicables, recopiée exactement (ex : 'F 190 315', 'HCE 2504'). null si absente")
#     edition_cg: Optional[str] = Field(description="Date ou édition des Conditions Générales (ex : '04/2025', 'janvier 2023'). null si absente")
#     formule: Optional[str] = Field(description="Nom de la formule souscrite. null si absente")
#     garanties_souscrites: list[str] = Field(description="Garanties marquées comme souscrites")
#     garanties_non_souscrites: list[str] = Field(description="Garanties marquées comme non souscrites")
#     franchise: Optional[str] = Field(description="Montant de la franchise générale, avec l'unité")
#     capital_mobilier: Optional[str] = Field(description="Capital mobilier assuré, avec l'unité")

# print(FicheCP.model_json_schema()["properties"].keys())

In [14]:
# from typing import Optional
# from pydantic import BaseModel, Field

# class FicheCP(BaseModel):
#     assureur: str = Field(max_length=80,
#         description="Nom de l'assureur tel qu'écrit dans le document")
#     reference_cg: Optional[str] = Field(max_length=30,
#         description="Code de référence des Conditions Générales, SANS la date (ex : 'F 190 315', 'HCE 2504'). null si absent")
#     edition_cg: Optional[str] = Field(max_length=30,
#         description="Date ou édition des Conditions Générales (ex : '04/2025', 'janvier 2023'). null si absente")
#     formule: Optional[str] = Field(max_length=50,
#         description="Nom de la formule souscrite. null si absente")

# print(FicheCP.model_json_schema()["properties"].keys())

In [15]:
from typing import Optional
from pydantic import BaseModel, Field
class FicheCP(BaseModel):
    assureur: str = Field(max_length=80,
        description="Nom de l'assureur tel qu'écrit dans le document")
    reference_cg: Optional[str] = Field(max_length=30,
        description="CODE alphanumérique de référence des Conditions Générales (ex : 'F 190 315', 'HCE 2504'). "
                    "Ce n'est NI le nom du produit NI une date. Mets null si le document ne donne pas de code.")
    edition_cg: Optional[str] = Field(max_length=30,
        description="Date d'édition des Conditions Générales, AVEC le mois et l'année si présents "
                    "(ex : '04/2025', 'janvier 2023'). null si absente.")
    formule: Optional[str] = Field(max_length=50,
        description="Nom court de la formule souscrite, tel qu'indiqué dans la section des garanties "
                    "(ex : 'Couverture Étendue', 'Formule Confort'). null si absente.")

In [ ]:
import time
from langchain_ollama import ChatOllama

consigne = """Tu extrais des informations de conditions particulières (CP) d'assurance habitation.
Règles :
- Recopie les valeurs exactement comme dans le document, sans les reformuler.
- Si une information est absente du document, mets null. N'invente jamais.

Document :
{texte}"""
llm = ChatOllama(model=LLM_MODEL, temperature=0)
extracteur = llm.with_structured_output(FicheCP, method="json_schema")
fiche = []
for i ,  texte in enumerate([texte_cp3, texte_cp4]):
    debut = time.time()
    fiche.append(extracteur.invoke(consigne.format(texte=texte)))
    print(f"Fiche {i+1} :")
    print(f"Extraction en {time.time() - debut:.1f} s\n")
    print(fiche[i].model_dump_json(indent=2))

Fiche 1 :
Extraction en 42.8 s

{
  "assureur": "Sogessur (distribué par Société Générale)",
  "reference_cg": "Réf. F 190 315 - 04/2025",
  "edition_cg": "04/2025",
  "formule": "Formule Confort"
}
Fiche 2 :
Extraction en 33.5 s

{
  "assureur": "GMF",
  "reference_cg": "Conditions Générales Habitaion",
  "edition_cg": "janvier 2023",
  "formule": "Domo Pass"
}


In [17]:
print(FicheCP.model_fields["reference_cg"].description)

CODE alphanumérique de référence des Conditions Générales (ex : 'F 190 315', 'HCE 2504'). Ce n'est NI le nom du produit NI une date. Mets null si le document ne donne pas de code.


In [ ]:
import json, re, unicodedata

catalogue = json.loads(CATALOGUE.read_text(encoding="utf-8"))

MOIS = {"janvier": "01", "fevrier": "02", "mars": "03", "avril": "04", "mai": "05", "juin": "06",
        "juillet": "07", "aout": "08", "septembre": "09", "octobre": "10", "novembre": "11", "decembre": "12"}

def normaliser(s):
    """'Société Générale' -> 'societegenerale' ; 'F 190 315 - 04/2025' -> 'f190315042025'."""
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode().lower()   # enlève les accents
    for nom, num in MOIS.items():
        s = s.replace(nom, num)                                  # 'janvier 2023' -> '01 2023'
    return re.sub(r"[^a-z0-9]", "", s)                           # garde seulement lettres et chiffres
def lignes_cg(texte_cp):
    """Les lignes de la CP qui mentionnent les Conditions Générales."""
    return " ".join(l for l in texte_cp.splitlines() if "conditions générales" in l.lower())
def trouver_contrat(fiche):
    assureur = normaliser(fiche.assureur)
    ref_fiche = normaliser(fiche.reference_cg)
    ref_et_edition = normaliser(f"{fiche.reference_cg} {fiche.edition_cg}")

    candidats = set()
    for e in catalogue:
        # print("maintenant :", e["contrat_id"])
        # print("assureur :", e["assureur"], "alias :", e["alias"], "reference :", e["reference"], "edition :", e["edition"] , "type_doc :", e["type_doc"])
        if e["type_doc"] != "CG":
            continue
        # 1. Le bon assureur ? (nom officiel ou un de ses alias)
        if not any(normaliser(nom) in assureur for nom in [e["assureur"]] + e["alias"]):
            continue
        # 2. La bonne référence ? (si le catalogue en a une)
        if e["reference"] and normaliser(e["reference"]) not in ref_fiche:
            continue
        # 3. La bonne édition ? (si le catalogue en a une)
        if e["edition"] and normaliser(e["edition"]) not in ref_et_edition:
            continue
        candidats.add(e["contrat_id"])
        print("gardée :", e["contrat_id"])


    # Un seul candidat : on l'a. Zéro ou plusieurs : on refuse de deviner.
    return candidats.pop() if len(candidats) == 1 else None

for i, f in enumerate(fiche):
    contrat_id = trouver_contrat(f)
    print(f"Fiche {i+1} : contrat_id = {contrat_id}")

gardée : sg_F190315_2025-04
Fiche 1 : contrat_id = sg_F190315_2025-04
gardée : gmf_domopass_2023-01
Fiche 2 : contrat_id = gmf_domopass_2023-01


In [19]:
for i, f in enumerate(fiche):
    contrat_id = trouver_contrat(f)
    print(f"Fiche {i+1} : contrat_id = {contrat_id}")

# print(normaliser("Sogessur (distribué par Société Générale)"))
# print(normaliser("F 190 315 - 04/2025"))
# print(normaliser("janvier 2023"), normaliser("01/2023"))

gardée : sg_F190315_2025-04
Fiche 1 : contrat_id = sg_F190315_2025-04
gardée : gmf_domopass_2023-01
Fiche 2 : contrat_id = gmf_domopass_2023-01


In [20]:
# La bonne réponse pour chaque CP (on la connaît, puisqu'on a écrit les CP)
attendu = {
    "cp_01_luko_etendue.pdf":                "luko_hce_2504",
    "cp_02_luko_minimum.pdf":                "luko_hmin_2504",
    "cp_03_sogessur_F190315.pdf":            "sg_F190315_2025-04",
    "cp_04_gmf_domopass.pdf":                "gmf_domopass_2023-01",
    "cp_05_matmut_residence_principale.pdf": "matmut_rp",
    "cp_06_axa_hors_base.pdf":               None,      # absent de la base : on attend un refus
}

reussites = 0
temps = []
for nom, contrat_attendu in attendu.items():
    texte = PdfReader(DOSSIER_CP / nom).pages[0].extract_text()

    debut = time.time()
    fiche = extracteur.invoke(consigne.format(texte=texte))
    temps.append(time.time() - debut)

    contrat_trouve = trouver_contrat(fiche)
    ok = contrat_trouve == contrat_attendu
    reussites += ok

    print(f"{'✅' if ok else '❌'} {nom:40} attendu={contrat_attendu}  trouvé={contrat_trouve}")
    if not ok:   # en cas d'échec, on affiche la fiche pour comprendre pourquoi
        print("    fiche :", fiche.model_dump())

print(f"\nRéussite : {reussites}/{len(attendu)}  |  temps moyen d'extraction : {sum(temps) / len(temps):.1f} s")

gardée : luko_hce_2504
✅ cp_01_luko_etendue.pdf                   attendu=luko_hce_2504  trouvé=luko_hce_2504
gardée : luko_hmin_2504
✅ cp_02_luko_minimum.pdf                   attendu=luko_hmin_2504  trouvé=luko_hmin_2504
gardée : sg_F190315_2025-04
✅ cp_03_sogessur_F190315.pdf               attendu=sg_F190315_2025-04  trouvé=sg_F190315_2025-04
gardée : gmf_domopass_2023-01
✅ cp_04_gmf_domopass.pdf                   attendu=gmf_domopass_2023-01  trouvé=gmf_domopass_2023-01
gardée : matmut_rp
✅ cp_05_matmut_residence_principale.pdf    attendu=matmut_rp  trouvé=matmut_rp
✅ cp_06_axa_hors_base.pdf                  attendu=None  trouvé=None

Réussite : 6/6  |  temps moyen d'extraction : 37.1 s


In [21]:
from typing import Literal

cg = [e for e in catalogue if e["type_doc"] == "CG"]
ids_valides = [e["contrat_id"] for e in cg] + ["AUCUN"]

class Choix(BaseModel):
    justification: str = Field(max_length=300,
        description="En une ou deux phrases : quel assureur, quelle référence et quelle édition tu as lus dans les CP")
    contrat_id: Literal[tuple(ids_valides)]

liste_contrats = "\n".join(
    f"- {e['contrat_id']} : {e['assureur']} (alias : {', '.join(e['alias']) or 'aucun'}), produit {e['produit']}, "
    f"formule {e['formule']}, référence {e['reference']}, édition {e['edition']}"
    for e in cg)

consigne_choix = """Voici les Conditions Générales disponibles dans notre base :
{contrats}

Voici les Conditions Particulières d'un client :
{cp}

Quelles Conditions Générales s'appliquent à ce client ?
Règles : l'assureur, la référence et l'édition doivent correspondre. Si aucune ligne ne correspond exactement, réponds AUCUN.
Ne choisis jamais une autre version ou un autre assureur par défaut."""

choisisseur = llm.with_structured_output(Choix, method="json_schema")
print(liste_contrats)

- luko_hce_2504 : Luko (alias : Luko by Allianz Direct, Allianz Direct), produit Habitation, formule Couverture Étendue, référence HCE, édition 2504
- luko_hmin_2504 : Luko (alias : Luko by Allianz Direct, Allianz Direct), produit Habitation, formule Minimum Légal, référence HMIN, édition 2504
- sg_D190315_2024-01 : Société Générale (alias : Sogessur, SG), produit Assurance Habitation, formule None, référence D 190 315, édition 01/2024
- sg_F190315_2025-04 : Société Générale (alias : Sogessur, SG), produit Assurance Habitation, formule None, référence F 190 315, édition 04/2025
- sg_G190315_2026-01 : Société Générale (alias : Sogessur, SG), produit Assurance Habitation, formule None, référence G 190 315, édition 01/2026
- gmf_domopass_2023-01 : GMF (alias : La Garantie Mutuelle des Fonctionnaires), produit Domo Pass, formule None, référence None, édition 01/2023
- matmut_rp : Matmut (alias : aucun), produit Habitation Résidence Principale, formule None, référence None, édition None


In [ ]:
score_regles, score_llm = 0, 0
for nom, contrat_attendu in attendu.items():
    texte = PdfReader(DOSSIER_CP / nom).pages[0].extract_text()

    # Méthode 1 : LLM extrait la fiche + règles
    via_regles = trouver_contrat(extracteur.invoke(consigne.format(texte=texte)))

    # Méthode 2 : le LLM choisit directement
    choix = choisisseur.invoke(consigne_choix.format(contrats=liste_contrats, cp=texte))
    via_llm = None if choix.contrat_id == "AUCUN" else choix.contrat_id

    score_regles += via_regles == contrat_attendu
    score_llm += via_llm == contrat_attendu
    print(f"{nom:40} attendu={contrat_attendu}")
    print(f"   règles : {via_regles}   |   LLM : {via_llm}  ({choix.justification})")

print(f"\nRègles : {score_regles}/6   |   LLM : {score_llm}/6")

gardée : luko_hce_2504
cp_01_luko_etendue.pdf                   attendu=luko_hce_2504
   règles : luko_hce_2504   |   LLM : luko_hce_2504  (Le client a souscrit un contrat avec Luko by Allianz Direct, formule Couverture Étendue (réf. HCE 2504), et la Conditions Générales correspondante est **luko_hce_2504** (assureur : Luko by Allianz Direct, référence : HCE, édition : 2504). Aucune autre condition générale ne correspond exactement aux )
gardée : luko_hmin_2504
cp_02_luko_minimum.pdf                   attendu=luko_hmin_2504
   règles : luko_hmin_2504   |   LLM : luko_hmin_2504  (Le contrat mentionne clairement l'assureur 'Luko by Allianz Direct' et la formule 'Minimum Légal' avec la référence HMIN 2504. La correspondance exacte est trouvée dans la ligne : luko_hmin_2504 (assureur : Luko by Allianz Direct, formule : Minimum Légal, référence HMIN, édition 2504).)


In [17]:
variantes = [
    # (CP de départ,                 texte d'origine,         remplacé par,             contrat attendu)
    ("cp_03_sogessur_F190315.pdf", "F 190 315 - 04/2025", "D 190 315 - 01/2024", "sg_D190315_2024-01"),  # autre version, présente
    ("cp_03_sogessur_F190315.pdf", "F 190 315 - 04/2025", "G 190 315 - 01/2026", "sg_G190315_2026-01"),  # autre version, présente
    ("cp_03_sogessur_F190315.pdf", "F 190 315 - 04/2025", "F 190 315 - 09/2022", None),                  # version ABSENTE
    ("cp_01_luko_etendue.pdf",     "HCE 2504",            "HCE 2410",            None),                  # édition ABSENTE
    ("cp_04_gmf_domopass.pdf",     "janvier 2023",        "mars 2021",           None),                  # édition ABSENTE
]

for nom, avant, apres, contrat_attendu in variantes:
    texte = PdfReader(DOSSIER_CP / nom).pages[0].extract_text()
    assert avant in texte, f"'{avant}' introuvable dans {nom}"   # on vérifie que le remplacement a bien lieu
    texte = texte.replace(avant, apres)

    via_regles = trouver_contrat(extracteur.invoke(consigne.format(texte=texte)))
    choix = choisisseur.invoke(consigne_choix.format(contrats=liste_contrats, cp=texte))
    via_llm = None if choix.contrat_id == "AUCUN" else choix.contrat_id

    print(f"{apres:22} attendu={contrat_attendu}")
    print(f"   règles : {via_regles} {'✅' if via_regles == contrat_attendu else '❌'}"
          f"   |   LLM : {via_llm} {'✅' if via_llm == contrat_attendu else '❌'}")

gardée : sg_D190315_2024-01
D 190 315 - 01/2024    attendu=sg_D190315_2024-01
   règles : sg_D190315_2024-01 ✅   |   LLM : sg_D190315_2024-01 ✅
gardée : sg_G190315_2026-01
G 190 315 - 01/2026    attendu=sg_G190315_2026-01
   règles : sg_G190315_2026-01 ✅   |   LLM : sg_D190315_2024-01 ❌
F 190 315 - 09/2022    attendu=None
   règles : None ✅   |   LLM : sg_F190315_2025-04 ❌
HCE 2410               attendu=None
   règles : None ✅   |   LLM : luko_hce_2504 ❌
mars 2021              attendu=None
   règles : None ✅   |   LLM : gmf_domopass_2023-01 ❌
